# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [19]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [20]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('ZAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = "glm-4.7-flash"
client = OpenAI(
    api_key=api_key,
    base_url="https://api.z.ai/api/paas/v4/"
)

An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


In [21]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GLM-5 figure out which links are relevant

### Use a call to glm-5 to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [22]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [23]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [24]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [25]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [26]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'social media', 'url': 'https://www.linkedin.c

In [27]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [28]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling glm-4.7-flash
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company product',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'project', 'url': 'https://edwarddonner.com/proficient/'}]}

In [29]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling glm-4.7-flash
Found 15 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'open source page', 'url': 'https://github.com/huggingface'},
  {'type': 'social media page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'social media page',
   'url': 'https://www.linkedin.com/company/hugging

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [32]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        link_url = link["url"]
        if not link_url.startswith("http"):
            print(f"Skipping invalid URL: {link_url}")
            continue
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link_url)
    return result

In [33]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling glm-4.7-flash
Found 3 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
4 days ago
•
587k
•
816
Qwen/Qwen3.5-27B
Updated
6 days ago
•
260k
•
517
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
3 days ago
•
503k
•
453
Qwen/Qwen3.5-122B-A10B
Updated
about 20 hours ago
•
134k
•
373
Qwen/Qwen3.5-397B-A17B
Updated
7 days ago
•
1.14M
•
1.18k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.78k
Qwen Image Multiple Angles 3D Camera
🎥
1.78k
Change the camera angle of a photo with AI
Running
on
Zero
Featured
281
Omni Video Factory
🏆
281
text to video, image to video, video extend
Running
on


In [43]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [35]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [36]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling glm-4.7-flash
Found 7 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n4 days ago\n•\n587k\n•\n816\nQwen/Qwen3.5-27B\nUpdated\n6 days ago\n•\n260k\n•\n517\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n3 days ago\n•\n503k\n•\n453\nQwen/Qwen3.5-122B-A10B\nUpdated\nabout 20 hours ago\n•\n134k\n•\n373\nQwen/Qwen3.5-397B-A17B\nUpdated\n7 days ago\n•\n1.14M\n•\n1.18k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.78k\nQwen Image 

In [37]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="glm-4.7-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [38]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling glm-4.7-flash
Found 5 relevant links


# Hugging Face: The AI Community Building the Future

**Hugging Face** is the central collaboration platform for the machine learning community. We empower the next generation of engineers, scientists, and end users to learn, collaborate, and share their work to build an open and ethical AI future together.

### Our Platform: The Hub
At the heart of our operations is the **Hugging Face Hub**, a vast ecosystem where users can share, explore, discover, and experiment with open-source machine learning.

*   **Scale:** We host **2M+ models**, **500k+ datasets**, and **1M+ applications**.
*   **Modality:** We support every modality, from text and image to video, audio, and 3D.
*   **Speed:** We provide the open-source stack that allows the community to move faster than ever before.

### Our Community & Mission
We are at the heart of the AI revolution, driven by a fast-growing community and a talented science team exploring the edge of technology. Our brand universe is built on the belief that open collaboration is key to innovation. We focus on diverse fields including NLP, Audio, Computer Vision (CV), Reinforcement Learning (RL), and Ethics.

### Careers at Hugging Face
We are looking for talent to join our team at the forefront of the AI revolution. Whether you are a machine learning engineer, a researcher, or a developer, we invite you to explore our **current openings**. Join a culture that values creativity, open-source contribution, and building the future of technology.

*Join us in building the AI community that powers the world.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [41]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="glm-5",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [42]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling glm-4.7-flash
Found 9 relevant links


# Hugging Face
## The AI Community Building the Future

---

### Who We Are

Hugging Face is the world's leading AI platform where the machine learning community collaborates on models, datasets, and applications. We are building the home of machine learning — a place where anyone can create, discover, and collaborate on ML projects.

---

### Our Platform

**Models** — Browse over **2 million models** from cutting-edge language models to image, video, audio, and 3D generators. Our community shares and improves models together, making AI accessible to all.

**Datasets** — Access **500,000+ datasets** to train and evaluate your machine learning projects. From text corpora to multimodal collections, find the data you need.

**Spaces** — Explore **1 million+ AI applications** running directly in your browser. Demo your work, test new ideas, and showcase innovations with the community.

---

### What We Offer

**For Individuals**
- Host unlimited public models, datasets, and applications
- Build your ML portfolio and share your work with the world
- Collaborate with a global community of researchers and developers
- Access our open-source stack to move faster

**For Teams & Enterprise**
- Team plans starting at **$20/user/month**
- Enterprise-grade security and access controls
- Single Sign-On (SSO) integration
- Regional data management and audit logs
- Dedicated support and flexible contract options

---

### Why Hugging Face?

**Open Source First** — We believe in the power of open collaboration. Our open-source stack helps you move faster and build better.

**All Modalities** — Work with text, image, video, audio, and even 3D — all in one platform.

**Community Driven** — Join millions of developers, researchers, and companies building the future of AI together.

---

### Join Us

Whether you're a researcher pushing the boundaries of AI, a developer building the next breakthrough app, or an enterprise scaling AI across your organization — Hugging Face is your home for machine learning.

**Get Started Today** — Sign up for free and become part of the AI community building the future.

---

*Visit us at huggingface.co*

In [44]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling glm-4.7-flash
Found 5 relevant links
Skipping invalid URL: /huggingface
Skipping invalid URL: /blog


# 🤗 Hugging Face
## *The AI Community Building the Future*

---

### Who Are We?

We're the reason your machine learning models don't live in lonely isolation on your laptop. Hugging Face is where the ML community comes to collaborate, share, and occasionally argue about transformer architectures at 2 AM.

Founded in 2016 and headquartered in Paris (yes, we do have excellent croissants in the office), we've grown to 51-200 employees who are passionately devoted to democratizing artificial intelligence—one pull request at a time.

---

### What We Actually Do

**We're basically GitHub for AI, but with more emojis.**

- **2 Million+ Models** — Need a text generator? An image creator? A 3D camera angle changer? We've got models for that.
- **500,000+ Datasets** — More data than you can shake a GPU at.
- **1 Million+ Applications** — "Spaces" where AI comes to play, demo, and impress your boss.

Text, image, video, audio, 3D—we support all modalities. If it can be learned by a machine, someone's probably already uploaded it.

---

### Our Vibe

We're **open source enthusiasts**, **community builders**, and **AI democrats** (not the political kind—the "AI should be accessible to everyone" kind).

We host hackathons with PyTorch. We build tools like Gradio that let you create rich text editors in a single Python file. We believe the future of coding might just be "vibe coding"—where you prompt your way to working components.

**Company Culture in a Nutshell:**
- 🇫🇷 Parisian headquarters = mandatory sophistication
- 🤝 Community-first mentality
- 🔓 Obsessively open source
- 🚀 Move fast and share things

---

### Who Uses Hugging Face?

Everyone from solo developers training their first neural network to enterprise teams deploying production AI systems. If you've typed `from transformers import` in Python, you've already met us.

---

### Want to Join Us?

We're a privately held company on a mission to solve and democratize artificial intelligence through natural language. We're looking for people who:

- Get excited about open source
- Think "democratizing AI" sounds like a pretty good life goal
- Can handle the occasional heated debate about model architectures
- Don't mind working alongside people who name their company after a hugging emoji

Check our current openings—we're always looking for fellow AI enthusiasts to join the hug.

---

### The Bottom Line

We're building the platform where machine learning happens together. Whether you're browsing trending models, hosting your dataset, or building the next breakthrough AI app—we're your home on the internet.

**Come for the models. Stay for the community.**

---

*🤗 Hugging Face — The AI community building the future.*

*Paris | Open Source | 2016–Present*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>